In [2]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from itertools import combinations
import os
from scipy import sparse

In [1]:
cd ./adatas/PB

d:\Projects\Geneformer\adatas\PB


d:\Gaol\miniconda3\envs\geneformer_env\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [10]:
ls

 Volume in drive D has no label.
 Volume Serial Number is 4EE5-FFF2

 Directory of d:\Projects\Geneformer\adatas\PB

02/24/2026  12:33 AM    <DIR>          .
02/11/2026  09:30 PM    <DIR>          ..
01/04/2025  11:57 AM     3,324,271,701 05b-aml-PB-annotated-complete.rds
02/24/2026  12:33 AM    <DIR>          DE_by_cluster
02/24/2026  12:19 AM        45,983,512 PB_HC_HSCLSC.h5ad
02/23/2026  11:08 PM        45,789,288 PB_RR_post_nres_HSCLSC.h5ad
02/23/2026  11:18 PM        38,231,452 PB_RR_post_nres_LE.h5ad
02/23/2026  11:08 PM        45,739,922 PB_RR_post_nres_Megakaryocyte.h5ad
02/23/2026  11:08 PM        62,087,862 PB_RR_post_nres_MPPCLP.h5ad
02/23/2026  11:08 PM        37,481,548 PB_RR_post_nres_MPPpreB.h5ad
02/23/2026  11:08 PM        37,552,704 PB_RR_post_nres_progenitor.h5ad
02/24/2026  12:06 AM        33,422,860 PB_RR_pre_nres_ASDC.h5ad
02/24/2026  12:06 AM        30,410,264 PB_RR_pre_nres_HSCLSC.h5ad
02/24/2026  12:06 AM        30,306,304 PB_RR_pre_nres_LE.h5ad
02/24/2026  12:

In [ ]:
组间对比
pre_res_PolychromaticAMLvsHC_NK
pre_res_PolychromaticAMLvsRR_post_nres_BasophilicAML
pre_res_PolychromaticAMLvsHC_post_res_PolychromaticAML
pre_res_PolychromaticAMLvsHC_pre_nres_DCAM


跨组对比
pre_res_PolychromaticLEvsHC_RBCLE
pre_res_PolychromaticLEvsRR_post_nres_PolychromaticLE
pre_res_PolychromaticLEvsRR_post_nres_BasophilicLE
pre_res_PolychromaticLEvsRR_post_nres_unknownLE
pre_res_PolychromaticLEvsRR_post_res_RBCLE
pre_res_PolychromaticLEvsRR_pre_nres_PolychromaticLE
pre_res_PolychromaticLEvsRR_pre_nres_RBCLE
pre_res_PolychromaticLEvsRR_pre_res_PolychromaticLE

HC_RBCLEvsRR_post_nres_PolychromaticLE
HC_RBCLEvsRR_post_nres_BasophilicLE
HC_RBCLEvsRR_post_nres_unknownLE
HC_RBCLEvsRR_post_res_RBCLE
HC_RBCLEvsRR_pre_nres_PolychromaticLE
HC_RBCLEvsRR_pre_nres_RBCLE
HC_RBCLEvsRR_pre_res_PolychromaticLE


EE系
HC_PolychromaticEEvspost_nres_macrophageEE
HC_PolychromaticEEvspre_nres_RBCEE
HC_PolychromaticEEvspre_res_PolychromaticEE
pre_res_PolychromaticEEvspost_nres_macrophageEE
pre_res_PolychromaticEEvspre_nres_RBCEE

In [ ]:
import os
import itertools
from collections import defaultdict

# 你的 h5ad 文件列表（可以用 os.listdir 读目录）
files = [
    "RR_HC_EarlyErythroidNormal.h5ad",
    "RR_HC_LateErythroidNormal.h5ad",
    "RR_post_nres_ErythroidNormal.h5ad",
    "RR_post_nres_LateErythroidAML.h5ad",
    "RR_post_nres_LateErythroidNormal.h5ad",
    "RR_post_nres_macrophageNormal.h5ad",
    "RR_post_nres_unknownNormal.h5ad",
    "RR_post_res_LateErythroidAML.h5ad",
    "RR_post_res_LateErythroidNormal.h5ad",
    "RR_pre_nres_cDC2AML.h5ad",
    "RR_pre_nres_LateErythroidNormal.h5ad",
    "RR_pre_res_EarlyErythroidNormal.h5ad",
    "RR_pre_res_LateErythroidAML.h5ad",
    "RR_pre_res_LateErythroidNormal.h5ad",
]

# 解析文件名
# RR_<group>_<celltype>.h5ad
entries = []
for f in files:
    name = f.replace(".h5ad", "")
    _, group, celltype = name.split("_", 2)
    entries.append((group, celltype, name))

# 按 group 和 celltype 归类
by_group = defaultdict(list)
by_celltype = defaultdict(list)

for group, celltype, name in entries:
    by_group[group].append((celltype, name))
    by_celltype[celltype].append((group, name))

clusters_to_test = []

# =========================
# 1) 组内：同组不同细胞类型两两对比
# =========================
for group, items in by_group.items():
    for (ct1, n1), (ct2, n2) in itertools.combinations(items, 2):
        tag = f"{group}_{ct1}__vs__{group}_{ct2}"
        clusters_to_test.append((tag, n1, n2))

# =========================
# 2) 跨组：同细胞类型对比（以 pre_res 为 baseline）
# =========================
baseline_group = "pre_res"

for celltype, items in by_celltype.items():
    base = [name for g, name in items if g == baseline_group]
    others = [(g, name) for g, name in items if g != baseline_group]

    if len(base) == 0:
        continue  # 这个细胞类型没有 pre_res，跳过

    base_name = base[0]

    for g, other_name in others:
        tag = f"{baseline_group}_{celltype}__vs__{g}_{celltype}"
        clusters_to_test.append((tag, base_name, other_name))

# 看看结果
print("Total comparisons:", len(clusters_to_test))
for x in clusters_to_test:
    print(x)


Total comparisons: 7
('RR_pre_res_EarlyErythroidNormalvsRR_pre_res_LateErythroidAML', 'RR_pre_res_EarlyErythroidNormal', 'RR_pre_res_LateErythroidAML')
('RR_pre_res_EarlyErythroidNormalvsRR_pre_res_LateErythroidNormal', 'RR_pre_res_EarlyErythroidNormal', 'RR_pre_res_LateErythroidNormal')
('RR_pre_res_LateErythroidAMLvsRR_post_nres_LateErythroidAML', 'RR_pre_res_LateErythroidAML', 'RR_post_nres_LateErythroidAML')
('RR_pre_res_LateErythroidAMLvsRR_post_res_LateErythroidAML', 'RR_pre_res_LateErythroidAML', 'RR_post_res_LateErythroidAML')
('RR_pre_res_LateErythroidNormalvsRR_post_nres_LateErythroidNormal', 'RR_pre_res_LateErythroidNormal', 'RR_post_nres_LateErythroidNormal')
('RR_pre_res_LateErythroidNormalvsRR_post_res_LateErythroidNormal', 'RR_pre_res_LateErythroidNormal', 'RR_post_res_LateErythroidNormal')
('RR_pre_res_LateErythroidNormalvsRR_pre_nres_LateErythroidNormal', 'RR_pre_res_LateErythroidNormal', 'RR_pre_nres_LateErythroidNormal')


In [35]:
clusters_to_test = [
    # ================= 1) 组内对比 (Baseline: LateErythroidNormal) =================
    # 逻辑：在同一组内，以 LateErythroidNormal 为基准对比其他细胞
    # 格式：("显示名称", "Baseline(LateNormal)", "测试组(其他)")

    # post_nres 组
    ("post_nres_MPPCLP_vs_post_nres_MPPpreB", 
     "PB_RR_post_nres_MPPCLP", "PB_RR_post_nres_MPPpreB"),

    # pren_res 组
    ("pre_res_MPPCLP_vs_pre_res_MPPpreB", 
     "PB_RR_pre_nres_MPPCLP", "PB_RR_pre_nres_MPPpreB"),
    ("pre_res_MPPCLP_vs_pre_res_ASDC", 
     "PB_RR_pre_nres_MPPCLP", "PB_RR_pre_nres_ASDC"),


    # ================= 2) 跨组同类型对比 (Baseline: pre_res) =================
    # 逻辑：针对同一种细胞，以 pre_res 组为基准对比其他组
    # 格式：("显示名称", "Baseline(pre_res)", "测试组(其他组)")

    # LateErythroidNormal 跨组
    ("pre_nres_LateErythroidNormal_vs_post_nres_LateErythroidNormal", 
     "PB_RR_pre_nres_LE", "PB_RR_post_nres_LE"),

    # HSCLSC 跨组
    ("HC_HSCLSC_vs_post_nres_HSCLSC", 
     "PB_HC_HSCLSC", "PB_RR_post_nres_HSCLSC"),
    ("HC_HSCLSC_vs_pre_nres_HSCLSC", 
     "PB_HC_HSCLSC", "PB_RR_pre_nres_HSCLSC"),
    # ASDC 跨组
    ("pre_res_ASDC_vs_pre_nres_ASDC", 
     "PB_RR_pre_res_ASDC", "PB_RR_pre_nres_ASDC"),

     # MPPCLP 跨组
    ("pre_nres_MPPCLP_vs_post_nres_MPPCLP", 
     "PB_RR_pre_nres_MPPCLP", "PB_RR_post_nres_MPPCLP"),
     # MPPPreB 跨组
    ("pre_nres_MPPPreB_vs_post_nres_MPPPreB", 
     "PB_RR_pre_nres_MPPpreB", "PB_RR_post_nres_MPPpreB"),
     # Megakaryocyte 跨组
    ("pre_nres_progenitor_vs_post_nres_Megakaryocyte", 
     "PB_RR_pre_nres_progenitor", "PB_RR_post_nres_Megakaryocyte"),
    ("pre_nres_progenitor_vs_post_nres_progenitor", 
     "PB_RR_pre_nres_progenitor", "PB_RR_post_nres_progenitor"),
]

In [11]:
clusters_to_test = [
    # post nres vs pre nres (same cell type)
    ("HSCLSC_post_vs_pre_nres", "PB_RR_pre_nres_HSCLSC", "PB_RR_post_nres_HSCLSC"),
    ("LE_post_vs_pre_nres",     "PB_RR_pre_nres_LE",     "PB_RR_post_nres_LE"),
    ("MPPCLP_post_vs_pre_nres", "PB_RR_pre_nres_MPPCLP", "PB_RR_post_nres_MPPCLP"),
    ("MPPpreB_post_vs_pre_nres","PB_RR_pre_nres_MPPpreB","PB_RR_post_nres_MPPpreB"),
    ("Prog_post_vs_pre_nres",   "PB_RR_pre_nres_progenitor","PB_RR_post_nres_progenitor"),

    # HC vs RR post nres
    ("HSCLSC_HC_vs_post_nres", "PB_HC_HSCLSC", "PB_RR_post_nres_HSCLSC"),

    # pre res vs pre nres (ASDC only, since你只有这个)
    ("ASDC_pre_res_vs_pre_nres", "PB_RR_pre_nres_ASDC", "PB_RR_pre_res_ASDC"),
]

In [39]:

# =========================================================
# 1) 参数（与你原逻辑一致；仅把 FC 设为 1.5）
# =========================================================
USE_LAYER = None        # e.g. "counts" / "lognorm"；不用就 None
USE_RAW = False         # 若要用 adata.raw，就 True（优先于 layers）
THRESHOLD = 0.0         # positive-only：只用 >threshold 的表达细胞
MIN_POS_CELLS = 20      # 每组至少多少表达细胞才纳入
USE_FDR = True          # 建议 True：多重检验
P_CUTOFF = 0.05
FDR_CUTOFF = 0.05

FC_CUTOFF = 2     # ✅ 你要的：FC > 1.5
FC_METHOD = "mean"      # "mean" 或 "median"
FC_PSEUDOCOUNT = 0.0    # 如担心除零，可设 1e-9

# 是否先做 normalize/log1p（强烈建议 True，除非你的 X 已经是 lognorm）
DO_PREPROCESS = False
TARGET_SUM = 1e4

# 输出目录
OUTDIR = "DE_by_cluster"
os.makedirs(OUTDIR, exist_ok=True)

DATA_DIR = r"d:\Projects\Geneformer\adatas\PB"

group_files = {
    os.path.splitext(f)[0]: os.path.join(DATA_DIR, f)
    for f in os.listdir(DATA_DIR)
    if f.lower().endswith(".h5ad")
}

# =========================================================
# 2) 工具函数
# =========================================================
def bh_fdr(pvals):
    """Benjamini–Hochberg FDR；返回与输入同长度的 qvals（np.array）"""
    pvals = np.asarray(pvals, dtype=float)
    qvals = np.full_like(pvals, np.nan, dtype=float)
    ok = np.isfinite(pvals)
    if ok.sum() == 0:
        return qvals
    pv = pvals[ok]
    order = np.argsort(pv)
    ranked = pv[order]
    m = len(ranked)
    q = ranked * m / (np.arange(1, m + 1))
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty_like(q)
    out[order] = q
    qvals[ok] = out
    return qvals

def get_matrix_and_genes(adata, use_layer=None, use_raw=False):
    """
    返回 (X, gene_names)
    X: cells x genes (csr_matrix or ndarray)
    gene_names: np.array of gene symbols
    """
    if use_raw:
        if adata.raw is None:
            return None, None
        X = adata.raw.X
        genes = np.asarray(adata.raw.var_names)
    elif use_layer is not None:
        if use_layer not in adata.layers:
            return None, None
        X = adata.layers[use_layer]
        genes = np.asarray(adata.var_names)
    else:
        X = adata.X
        genes = np.asarray(adata.var_names)

    if sparse.issparse(X):
        X = X.tocsr()
    else:
        X = np.asarray(X)
    return X, genes

def summary_stat(x, method="mean"):
    if method == "median":
        return float(np.median(x))
    return float(np.mean(x))

def conditional_mw_positive_only_allgenes_okonly(
    ad_res, ad_nres, res_name, nres_name,
    min_pos_cells=20, threshold=0.0,
    use_layer=None, use_raw=False,
    fc_method="mean",
    fc_pseudocount=0.0
):
    """
    对所有共有基因做 positive-only MWU：
      - 仅使用 >threshold 的细胞
      - 若任一组 positive cells < min_pos_cells：丢弃该基因
      - 计算 FC（nres/res）和 log2FC（nres/res），基于 positive-only 且用 mean/median
    """
    X1, genes1 = get_matrix_and_genes(ad_res, use_layer=use_layer, use_raw=use_raw)
    X2, genes2 = get_matrix_and_genes(ad_nres, use_layer=use_layer, use_raw=use_raw)
    if X1 is None or X2 is None:
        return pd.DataFrame()

    common = np.intersect1d(genes1, genes2, assume_unique=False)
    if common.size == 0:
        return pd.DataFrame()

    idx1 = pd.Index(genes1).get_indexer(common)
    idx2 = pd.Index(genes2).get_indexer(common)

    X1c = X1[:, idx1]
    X2c = X2[:, idx2]

    rows = []
    for j, gene in enumerate(common):
        if sparse.issparse(X1c):
            x1_all = X1c[:, j].toarray().ravel()
            x2_all = X2c[:, j].toarray().ravel()
        else:
            x1_all = np.asarray(X1c[:, j]).ravel()
            x2_all = np.asarray(X2c[:, j]).ravel()

        x1_pos = x1_all[x1_all > threshold]  # g1
        x2_pos = x2_all[x2_all > threshold]  # g2

        if (x1_pos.size < min_pos_cells) or (x2_pos.size < min_pos_cells):
            continue

        u, p = mannwhitneyu(x1_pos, x2_pos, alternative="two-sided")

        stat_g1 = summary_stat(x1_pos, method=fc_method)
        stat_g2 = summary_stat(x2_pos, method=fc_method)

        denom = stat_g1 + fc_pseudocount
        numer = stat_g2 + fc_pseudocount
        fc = (numer / denom) if denom > 0 else np.nan
        log2fc = np.log2(fc) if (np.isfinite(fc) and fc > 0) else np.nan

        rows.append({
            "gene": gene,
            "p_value": float(p),
            "U": float(u),
            f"{res_name}_n_pos": int(x1_pos.size),
            f"{nres_name}_n_pos": int(x2_pos.size),
            f"{fc_method}_g1_pos": float(stat_g1),
            f"{fc_method}_g2_pos": float(stat_g2),
            "FC_g2_over_g1": float(fc) if np.isfinite(fc) else np.nan,
            "log2FC_g2_over_g1": float(log2fc) if np.isfinite(log2fc) else np.nan,
            "mean_pos_diff(g1-g2)": float(np.mean(x1_pos) - np.mean(x2_pos)),
        })

    return pd.DataFrame(rows)

def maybe_preprocess(adata: sc.AnnData) -> sc.AnnData:
    """
    可选：对每个 adata 做 normalize_total + log1p（在 copy 上做）
    若你的 X 已经是 lognorm，就把 DO_PREPROCESS 设为 False
    """
    if not DO_PREPROCESS:
        return adata
    ad = adata.copy()
    sc.pp.normalize_total(ad, target_sum=TARGET_SUM)
    sc.pp.log1p(ad)
    return ad

# =========================================================
# 2.5) 检查：pairs 用到的 key 都存在
# =========================================================
missing = []
for name, g1, g2 in clusters_to_test:
    if g1 not in group_files:
        missing.append(g1)
    if g2 not in group_files:
        missing.append(g2)

if missing:
    raise KeyError(
        "Missing keys in group_files (check spelling / file names):\n"
        + "\n".join(sorted(set(missing)))
        + "\n\nAvailable keys example:\n"
        + "\n".join(list(sorted(group_files.keys()))[:10])
    )

# =========================================================
# 3) 读取数据
# =========================================================
adatas = {}
for k, fp in group_files.items():
    # 只读本次要用到的 key（可选：省内存）
    # 这里不改变结构太多：直接全部读也行，但更占内存
    pass

# 仅加载本次 pairs 需要的文件，避免一次性把 20 个都读进内存
needed_keys = sorted(set([g1 for _, g1, _ in clusters_to_test] + [g2 for _, _, g2 in clusters_to_test]))
for k in needed_keys:
    fp = group_files[k]
    if not os.path.exists(fp):
        raise FileNotFoundError(f"Missing file for {k}: {fp}")
    adatas[k] = sc.read_h5ad(fp)

# =========================================================
# 4) 主流程：逐 pair 跑 DE（g2 相对 g1 上调）
# =========================================================
all_sig_tables = []
log2fc_by_cluster = {}   # pair -> pd.Series(gene -> log2FC)
sig_gene_sets = {}       # pair -> set(sig_up_genes)

for pair_name, g1_key, g2_key in clusters_to_test:
    ad_g1 = maybe_preprocess(adatas[g1_key])
    ad_g2 = maybe_preprocess(adatas[g2_key])

    df = conditional_mw_positive_only_allgenes_okonly(
        ad_g1, ad_g2,
        res_name=g1_key, nres_name=g2_key,
        min_pos_cells=MIN_POS_CELLS,
        threshold=THRESHOLD,
        use_layer=USE_LAYER,
        use_raw=USE_RAW,
        fc_method=FC_METHOD,
        fc_pseudocount=FC_PSEUDOCOUNT
    )

    if df.shape[0] == 0:
        print(f"[WARN] {pair_name}: no genes passed MIN_POS_CELLS.")
        continue

    df["FDR_BH"] = bh_fdr(df["p_value"].values)

    # 显著性 + 上调(g2>g1) + FC>1.5
    if USE_FDR:
        sig_mask = (
            df["FDR_BH"].notna()
            & (df["FDR_BH"] < FDR_CUTOFF)
            & df["log2FC_g2_over_g1"].notna()
            & (df["log2FC_g2_over_g1"] > np.log2(FC_CUTOFF))
        )
    else:
        sig_mask = (
            df["p_value"].notna()
            & (df["p_value"] < P_CUTOFF)
            & df["log2FC_g2_over_g1"].notna()
            & (df["log2FC_g2_over_g1"] > np.log2(FC_CUTOFF))
        )

    up_mask = df["log2FC_g2_over_g1"].notna() & (df["log2FC_g2_over_g1"] > 0)
    sig_up = df[sig_mask & up_mask].copy()

    # 排序：更靠前=更显著
    sig_up = sig_up.sort_values(["FDR_BH", "p_value"], ascending=True)

    # 保存
    out_full = os.path.join(OUTDIR, f"{pair_name}__full_all_ok_genes.csv")
    out_sig  = os.path.join(OUTDIR, f"{pair_name}__SIG_up_in_{g2_key}__FCgt{FC_CUTOFF}.csv")
    df.to_csv(out_full, index=False)
    sig_up.to_csv(out_sig, index=False)

    print(f"[{pair_name}] ok_genes={df.shape[0]}  sig_up={sig_up.shape[0]}")
    print(f"  saved: {out_sig}")

    # 汇总用
    sig_up["pair"] = pair_name
    all_sig_tables.append(sig_up)

    log2fc_by_cluster[pair_name] = df.set_index("gene")["log2FC_g2_over_g1"]
    sig_gene_sets[pair_name] = set(sig_up["gene"].tolist())

# =========================================================
# 5) 合并输出：所有 pair 的显著上调基因清单
# =========================================================
combined_path = os.path.join(OUTDIR, f"ALL_pairs__SIG_up_combined__FCgt{FC_CUTOFF}.csv")
perpair_list_path = os.path.join(OUTDIR, f"ALL_pairs__SIG_up_gene_list_per_pair__FCgt{FC_CUTOFF}.csv")

if len(all_sig_tables) > 0:
    all_sig_df = pd.concat(all_sig_tables, ignore_index=True)
    all_sig_df.to_csv(combined_path, index=False)

    # 列表式：每个 pair 一行
    list_rows = []
    for pair_name, genes in sig_gene_sets.items():
        list_rows.append({
            "pair": pair_name,
            "n_sig_up_genes": len(genes),
            "genes_sig_up_in_g2": ";".join(sorted(list(genes)))
        })
    pd.DataFrame(list_rows).to_csv(perpair_list_path, index=False)

    print("\n[OK] Combined saved:")
    print(" ", combined_path)
    print(" ", perpair_list_path)
else:
    print("\n[WARN] No significant genes found in any pair. Nothing to combine.")

# =========================================================
# 6) （可选）把 combined 结果展开成 wide 表：gene × pair
#    这部分保持你原思路，但把路径和列名对齐
# =========================================================
# 你如果不需要 wide 输出，可以把下面整段注释掉

MAKE_WIDE = True

if MAKE_WIDE and os.path.exists(combined_path):
    df = pd.read_csv(combined_path)

    # ===== 只保留你关心的 pair（这里默认全保留；要筛就改 pairs_keep）=====
    # pairs_keep = [ ... ]  # 例如只留 AML 系等
    # df = df[df["pair"].isin(pairs_keep)]

    # ===== 设置多级索引：gene × pair =====
    df_idx = df.set_index(["gene", "pair"])

    # ===== 选择要展开的指标列 =====
    value_cols = [
        "p_value",
        # "FDR_BH",
        "FC_g2_over_g1",
        # "log2FC_g2_over_g1",
        f"{FC_METHOD}_g1_pos",
        f"{FC_METHOD}_g2_pos",
    ]
    value_cols = [c for c in value_cols if c in df.columns]

    # ===== unstack pair → 自动补 NaN =====
    df_wide = df_idx[value_cols].unstack("pair")

    # ===== 调整列顺序：pair 在前，metric 在后 =====
    df_wide = df_wide.swaplevel(0, 1, axis=1)
    df_wide = df_wide.sort_index(axis=1, level=0)

    # ===== 压平成单层列名 =====
    df_wide.columns = [f"{pair}__{metric}" for pair, metric in df_wide.columns]

    wide_path = os.path.join(OUTDIR, f"SIG_up_wide__FCgt{FC_CUTOFF}.csv")
    df_wide.to_csv(wide_path)
    print("\n[OK] Wide table saved:")
    print(" ", wide_path)


[post_nres_MPPCLP_vs_post_nres_MPPpreB] ok_genes=4253  sig_up=521
  saved: DE_by_cluster\post_nres_MPPCLP_vs_post_nres_MPPpreB__SIG_up_in_PB_RR_post_nres_MPPpreB__FCgt2.csv
[pre_res_MPPCLP_vs_pre_res_MPPpreB] ok_genes=8541  sig_up=0
  saved: DE_by_cluster\pre_res_MPPCLP_vs_pre_res_MPPpreB__SIG_up_in_PB_RR_pre_nres_MPPpreB__FCgt2.csv
[pre_res_MPPCLP_vs_pre_res_ASDC] ok_genes=5681  sig_up=36
  saved: DE_by_cluster\pre_res_MPPCLP_vs_pre_res_ASDC__SIG_up_in_PB_RR_pre_nres_ASDC__FCgt2.csv
[pre_nres_LateErythroidNormal_vs_post_nres_LateErythroidNormal] ok_genes=1905  sig_up=30
  saved: DE_by_cluster\pre_nres_LateErythroidNormal_vs_post_nres_LateErythroidNormal__SIG_up_in_PB_RR_post_nres_LE__FCgt2.csv
[HC_HSCLSC_vs_post_nres_HSCLSC] ok_genes=507  sig_up=15
  saved: DE_by_cluster\HC_HSCLSC_vs_post_nres_HSCLSC__SIG_up_in_PB_RR_post_nres_HSCLSC__FCgt2.csv
[HC_HSCLSC_vs_pre_nres_HSCLSC] ok_genes=452  sig_up=12
  saved: DE_by_cluster\HC_HSCLSC_vs_pre_nres_HSCLSC__SIG_up_in_PB_RR_pre_nres_HSCLSC__F

In [40]:

if MAKE_WIDE and os.path.exists(combined_path):
    df = pd.read_csv(combined_path)

    # ===== 只保留你关心的 pair（这里默认全保留；要筛就改 pairs_keep）=====
    # pairs_keep = [ ... ]  # 例如只留 AML 系等
    # df = df[df["pair"].isin(pairs_keep)]

    # ===== 设置多级索引：gene × pair =====
    df_idx = df.set_index(["gene", "pair"])

    # ===== 选择要展开的指标列 =====
    value_cols = [
        "p_value",
        # "FDR_BH",
        "FC_g2_over_g1",
        # "log2FC_g2_over_g1",
        f"{FC_METHOD}_g1_pos",
        f"{FC_METHOD}_g2_pos",
    ]
    value_cols = [c for c in value_cols if c in df.columns]

    # ===== unstack pair → 自动补 NaN =====
    df_wide = df_idx[value_cols].unstack("pair")

    # ===== 调整列顺序：pair 在前，metric 在后 =====
    df_wide = df_wide.swaplevel(0, 1, axis=1)
    df_wide = df_wide.sort_index(axis=1, level=0)

    # ===== 压平成单层列名 =====
    df_wide.columns = [f"{pair}__{metric}" for pair, metric in df_wide.columns]

    wide_path = os.path.join(OUTDIR, f"SIG_up_wide__FCgt{FC_CUTOFF}.csv")
    df_wide.to_csv(wide_path)
    print("\n[OK] Wide table saved:")
    print(" ", wide_path)


[OK] Wide table saved:
  DE_by_cluster\SIG_up_wide__FCgt2.csv


In [11]:
import scanpy as sc
import pandas as pd
import numpy as np
from scipy import sparse


# ====== BM (你现有的 RR/Comparison) ======
files = {
    "BM_HC_HSC": "./adatas/RR/Comparison/RR_HC_HSC.h5ad",
    "BM_HC_Megakaryocyte": "./adatas/RR/Comparison/RR_HC_Megakaryocyte.h5ad",

    "BM_pre_res_HSCLSC": "./adatas/RR/Comparison/RR_pre_res_HSCLSC.h5ad",
    "BM_pre_nres_HSCLSC": "./adatas/RR/Comparison/RR_pre_nres_HSCLSC.h5ad",
    "BM_post_nres_HSCLSC": "./adatas/RR/Comparison/RR_post_nres_HSCLSC.h5ad",

    "BM_pre_res_MPPPreB": "./adatas/RR/Comparison/RR_pre_res_MPPPreB.h5ad",
    "BM_pre_nres_MPPPreB": "./adatas/RR/Comparison/RR_pre_nres_MPPPreB.h5ad",

    "BM_pre_res_MPPCLP1": "./adatas/RR/Comparison/RR_pre_res_MPPCLP1.h5ad",
    "BM_pre_nres_MPPCLP1": "./adatas/RR/Comparison/RR_pre_nres_MPPCLP1.h5ad",
    "BM_post_nres_MPPCLP1": "./adatas/RR/Comparison/RR_post_nres_MPPCLP1.h5ad",
    "BM_post_nres_ASDC": "./adatas/RR/Comparison/RR_post_nres_ASDC.h5ad",
    "BM_post_nres_MPPpreB": "./adatas/RR/Comparison/RR_post_nres_MPPpreB.h5ad",

    "BM_pre_res_Megakaryocyte": "./adatas/RR/Comparison/RR_pre_res_Megakaryocyte.h5ad",
    "BM_pre_nres_Megakaryocyte": "./adatas/RR/Comparison/RR_pre_nres_Megakaryocyte.h5ad",
    "BM_post_nres_Megakaryocyte": "./adatas/RR/Comparison/RR_post_nres_Megakaryocyte.h5ad",
    "BM_post_nres_progenitor": "./adatas/RR/Comparison/RR_post_nres_progenitor.h5ad",
    # ================= PB cohorts (你已有的) =================
    "PB_HC_HSCLSC": r"d:\Projects\Geneformer\adatas\PB\PB_HC_HSCLSC.h5ad",
    "PB_RR_pre_nres_HSCLSC": r"d:\Projects\Geneformer\adatas\PB\PB_RR_pre_nres_HSCLSC.h5ad",
    "PB_RR_post_nres_HSCLSC": r"d:\Projects\Geneformer\adatas\PB\PB_RR_post_nres_HSCLSC.h5ad",

    "PB_RR_pre_nres_MPPCLP": r"d:\Projects\Geneformer\adatas\PB\PB_RR_pre_nres_MPPCLP.h5ad",
    "PB_RR_post_nres_MPPCLP": r"d:\Projects\Geneformer\adatas\PB\PB_RR_post_nres_MPPCLP.h5ad",

    "PB_RR_pre_nres_MPPpreB": r"d:\Projects\Geneformer\adatas\PB\PB_RR_pre_nres_MPPpreB.h5ad",
    "PB_RR_post_nres_MPPpreB": r"d:\Projects\Geneformer\adatas\PB\PB_RR_post_nres_MPPpreB.h5ad",

    "PB_RR_pre_nres_ASDC": r"d:\Projects\Geneformer\adatas\PB\PB_RR_pre_nres_ASDC.h5ad",
    # "PB_RR_post_nres_ASDC": r"d:\Projects\Geneformer\adatas\PB\PB_RR_post_nres_ASDC.h5ad",

    "PB_RR_pre_nres_LE": r"d:\Projects\Geneformer\adatas\PB\PB_RR_pre_nres_LE.h5ad",
    "PB_RR_post_nres_LE": r"d:\Projects\Geneformer\adatas\PB\PB_RR_post_nres_LE.h5ad",

    "PB_RR_pre_nres_progenitor": r"d:\Projects\Geneformer\adatas\PB\PB_RR_pre_nres_progenitor.h5ad",
    "PB_RR_post_nres_progenitor": r"d:\Projects\Geneformer\adatas\PB\PB_RR_post_nres_progenitor.h5ad",

    # "PB_RR_pre_nres_Megakaryocyte": r"d:\Projects\Geneformer\adatas\PB\PB_RR_pre_nres_Megakaryocyte.h5ad",
    "PB_RR_post_nres_Megakaryocyte": r"d:\Projects\Geneformer\adatas\PB\PB_RR_post_nres_Megakaryocyte.h5ad",
}
adatas = {k: sc.read_h5ad(v) for k, v in files.items()}

In [21]:
comparisons = {
    # ================= HSCLSC =================
    "HSCLSC": {
        "baseline": "BM_pre_res_HSCLSC",  # BM baseline（优先 pre_res）
        "targets": [
            "PB_HC_HSCLSC",
            "PB_RR_pre_nres_HSCLSC",
            "PB_RR_post_nres_HSCLSC",
        ],
    },

    # ================= MPPPreB =================
    "MPPPreB": {
        "baseline": "BM_pre_res_MPPPreB",
        "targets": [
            "PB_RR_pre_nres_MPPpreB",
            "PB_RR_post_nres_MPPpreB",
        ],
    },

    # ================= MPPCLP =================
    "MPPCLP1": {
        "baseline": "BM_pre_res_MPPCLP1",
        "targets": [
            "PB_RR_pre_nres_MPPCLP",
            "PB_RR_post_nres_MPPCLP",
        ],
    },

    # ================= ASDC =================
    # BM 里只有 post_nres_ASDC（没有 pre_res / HC）
    "ASDC": {
        "baseline": "BM_post_nres_ASDC",
        "targets": [
            "PB_RR_pre_nres_ASDC",
            # "PB_RR_post_nres_ASDC",
        ],
    },

    # ================= LateErythroidNormal =================
    # ⚠️ BM 里你没给 LE，如果其实有，请换成对应的 BM key
    # "LateErythroidNormal": {
    #     "baseline": "BM_pre_res_LateErythroidNormal",  # ⚠️ 占位：请换成真实 BM_LE
    #     "targets": [
    #         "PB_RR_pre_nres_LE",
    #         "PB_RR_post_nres_LE",
    #     ],
    # },

    # ================= Megakaryocyte =================
    "Megakaryocyte": {
        "baseline": "BM_pre_res_Megakaryocyte",
        "targets": [
            # "PB_RR_pre_nres_Megakaryocyte",
            "PB_RR_post_nres_Megakaryocyte",
        ],
    },

    # ================= progenitor =================
    # 没有 pre_res / HC，只能退而求其次
    "progenitor": {
        "baseline": "BM_post_nres_progenitor",
        "targets": [
            "PB_RR_pre_nres_progenitor",
            "PB_RR_post_nres_progenitor",
        ],
    },

    # ================= HSC =================
    # 只有 HC_HSC
    "HSC": {
        "baseline": "BM_HC_HSC",
        "targets": [
            "PB_HC_HSCLSC",  # 如果你有 PB_HSC，请换成那个
        ],
    },
}

In [23]:
genes = [
    "BCL11A",
    "NOTCH1",
    "BCL2L1",
    "NRIP1",
    "ATF1",
    "CDK6",
    "PRDX1",
    "AKT1",
    "AKT2",
    "AKT3",
    "ATG16L2",
    "DAPK1",
    "LRBA",
    "NCOA7",
    "NOL4L",
    "PRKACB",
    "CD74",
]

In [24]:
def mean_expression(adata, genes):
    present = [g for g in genes if g in adata.var_names]
    X = adata[:, present].X

    if sparse.issparse(X):
        mean_vals = np.asarray(X.mean(axis=0)).ravel()
    else:
        mean_vals = X.mean(axis=0)

    return pd.Series(mean_vals, index=present)
mean_expr = {}
for name, ad in adatas.items():
    mean_expr[name] = mean_expression(ad, genes)


In [25]:
rows = []

for group, cfg in comparisons.items():
    base = cfg["baseline"]
    base_mean = mean_expr[base]

    for tgt in cfg["targets"]:
        tgt_mean = mean_expr[tgt]

        common = base_mean.index.intersection(tgt_mean.index)
        ratio = tgt_mean[common] / base_mean[common]

        for g in common:
            rows.append({
                "group": group,
                "baseline": base,
                "target": tgt,
                "gene": g,
                "mean_baseline": base_mean[g],
                "mean_target": tgt_mean[g],
                "ratio_target_over_baseline": ratio[g],
                "log2_ratio": np.log2(ratio[g]) if ratio[g] > 0 else np.nan
            })

ratio_df = pd.DataFrame(rows)


In [26]:
ratio_df

,group,baseline,target,gene,mean_baseline,mean_target,ratio_target_over_baseline,log2_ratio
0,HSCLSC,BM_pre_res_HSCLSC,PB_HC_HSCLSC,BCL11A,0.741342,0.555556,0.749392,-0.416208
1,HSCLSC,BM_pre_res_HSCLSC,PB_HC_HSCLSC,NOTCH1,0.370128,0.111111,0.300197,-1.736021
2,HSCLSC,BM_pre_res_HSCLSC,PB_HC_HSCLSC,BCL2L1,0.353776,0.333333,0.942216,-0.085870
3,HSCLSC,BM_pre_res_HSCLSC,PB_HC_HSCLSC,NRIP1,1.659546,3.000000,1.807724,0.854174
4,HSCLSC,BM_pre_res_HSCLSC,PB_HC_HSCLSC,ATF1,0.126168,0.148148,1.174212,0.231693
...,...,...,...,...,...,...,...,...
199,HSC,BM_HC_HSC,PB_HC_HSCLSC,LRBA,0.524557,0.666667,1.270914,0.345866
200,HSC,BM_HC_HSC,PB_HC_HSCLSC,NCOA7,0.360945,0.777778,2.154836,1.107578
201,HSC,BM_HC_HSC,PB_HC_HSCLSC,NOL4L,0.106440,0.222222,2.087778,1.061968
202,HSC,BM_HC_HSC,PB_HC_HSCLSC,PRKACB,0.314208,1.148148,3.654097,1.869515


In [27]:
ratio_df.to_csv(
    "PB_gene_mean_expression_ratio_vs_baseline_all_groups.csv",
    index=False
)

print("Saved: gene_mean_expression_ratio_vs_baseline_all_groups.csv")


Saved: gene_mean_expression_ratio_vs_baseline_all_groups.csv
